In [11]:
import torch
from torchvision import transforms
from PIL import Image
from diffusers import UNet2DConditionModel, DDPMScheduler
import torch.nn.functional as F
import os

In [12]:
class ConditionalImageDataset(torch.utils.data.Dataset):
    def __init__(self, target_paths, condition_image_path, image_size=64):
        self.target_paths = target_paths
        self.condition_image_path = condition_image_path
        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,))
        ])

        # Load and preprocess the condition image once
        condition_image = Image.open(self.condition_image_path).convert("RGB")
        self.condition_image_tensor = self.transform(condition_image)

        # Add a batch dimension if necessary (e.g., [1, C, H, W])

    def __len__(self):
        return len(self.target_paths)

    def __getitem__(self, idx):
        target_image = Image.open(self.target_paths[idx]).convert("RGB")
        target_tensor = self.transform(target_image)
        return {"target": target_tensor, "condition": self.condition_image_tensor}


In [13]:
model = UNet2DConditionModel(
    sample_size=64,       # Image size
    in_channels=3,        # Channels for target image
    out_channels=3,       # Channels for predicted noise
    num_class_embeds=None,  # Not used for image condition
    cross_attention_dim=3  # Dimension for conditional image
)

In [14]:
noise_scheduler = DDPMScheduler(num_train_timesteps=1000)

def train(model, dataset, epochs=10, batch_size=16, lr=1e-4, device="cuda"):
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    model.to(device)

    for epoch in range(epochs):
        for batch in dataloader:
            target_images = batch["target"].to(device)
            condition_images = batch["condition"].to(device)

            # Sample random timesteps
            timesteps = torch.randint(0, noise_scheduler.num_train_timesteps, (target_images.size(0),), device=device).long()
            
            # Add noise to the images
            noise = torch.randn_like(target_images)
            noisy_images = noise_scheduler.add_noise(target_images, noise, timesteps)

            # Predict noise using the model
            predicted_noise = model(noisy_images, timesteps, encoder_hidden_states=condition_images).sample()

            # Compute loss
            loss = F.mse_loss(predicted_noise, noise)

            # Backpropagation
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        print(f"Epoch {epoch + 1}/{epochs} - Loss: {loss.item():.4f}")

In [15]:
import os

target_image_paths = sorted([
    os.path.join('/mnt/raid/home/ajarry/data/trainer/temp_train/true_train', f)
    for f in os.listdir('/mnt/raid/home/ajarry/data/trainer/temp_train/true_train')
    if f.endswith(('.jpg', '.png', '.jpeg'))  # Filter by image formats
])

condition_image_path = '/mnt/raid/home/ajarry/data/trainer/condition/003f1137-63cc-4047-a265-b4aef597f980_frame_12.png'



dataset = ConditionalImageDataset(target_image_paths, condition_image_path, image_size=128)
train(model, dataset, epochs=10, batch_size=16, lr=1e-4, device="cuda")


/mnt/raid/home/ajarry/.conda/envs/difmod/lib/python3.10/site-packages/diffusers/configuration_utils.py:140: FutureWarning: Accessing config attribute `num_train_timesteps` directly via 'DDPMScheduler' object attribute is deprecated. Please access 'num_train_timesteps' over 'DDPMScheduler's config object instead, e.g. 'scheduler.config.num_train_timesteps'.
  deprecate("direct config name access", "1.0.0", deprecation_message, standard_warn=False)


ValueError: too many values to unpack (expected 3)